In [20]:
# HW0: robust mega acquisition (Colab-first, git-safe)
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/egil10/stk-mat2011.git"
REPO_DIR = Path("/content/stk-mat2011") if IN_COLAB else Path.cwd().resolve().parents[1]

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        # keep notebook reproducible if you re-run later
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)

    # optional but useful when Colab VM is fresh
    req = REPO_DIR / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

# make scripts importable
SCRIPTS_DIR = REPO_DIR / "code" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

from p_duka import download_and_save

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_DIR={REPO_DIR}")
print(f"SCRIPTS_DIR={SCRIPTS_DIR}")

IN_COLAB=True
REPO_DIR=/content/stk-mat2011
SCRIPTS_DIR=/content/stk-mat2011/code/scripts


In [21]:
from datetime import datetime
import shutil
import pandas as pd

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

# Repo output path (where p_duka writes)
repo_processed = REPO_DIR / "code" / "data" / "processed"
repo_processed.mkdir(parents=True, exist_ok=True)

# Persistent cache on Drive (survives git pull/reclone)
drive_processed = Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed") if IN_COLAB else repo_processed
if IN_COLAB:
    drive_processed.mkdir(parents=True, exist_ok=True)


def month_list(start_ym: str, end_ym: str):
    """Inclusive YYYYMM range."""
    y0, m0 = int(start_ym[:4]), int(start_ym[4:6])
    y1, m1 = int(end_ym[:4]), int(end_ym[4:6])
    out = []
    y, m = y0, m0
    while (y < y1) or (y == y1 and m <= m1):
        out.append(f"{y}{m:02d}")
        if m == 12:
            y, m = y + 1, 1
        else:
            m += 1
    return out


def expected_files(pair: str, yyyymm: str):
    s = pair.replace("/", "").lower()
    return [
        f"{s}_dukascopy_bid_{yyyymm}.parquet",
        f"{s}_dukascopy_ask_{yyyymm}.parquet",
    ]


def sync_drive_to_repo():
    if not IN_COLAB:
        return
    copied = 0
    for fp in drive_processed.glob("*.parquet"):
        tgt = repo_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced Drive -> repo: {copied} new file(s)")


def sync_repo_to_drive():
    if not IN_COLAB:
        return
    copied = 0
    for fp in repo_processed.glob("*.parquet"):
        tgt = drive_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced repo -> Drive: {copied} new file(s)")


print(f"repo_processed={repo_processed}")
print(f"drive_processed={drive_processed}")

Mounted at /content/drive
repo_processed=/content/stk-mat2011/code/data/processed
drive_processed=/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed


In [22]:
# Fast acquisition: newest-first + chunked + periodic sync
PAIR = "EUR/USD"

# 1) Choose one chunk per run (recommended)
# Example quick chunk:
START_YYYYMM = "202201"
END_YYYYMM = "202212"
SYNC_EVERY = 1
MAX_FAIL_BEFORE_STOP = 3

# Pull persistent files into repo first
sync_drive_to_repo()

months = month_list(START_YYYYMM, END_YYYYMM)
months = list(reversed(months))  # newest first
print(f"Planned months: {len(months)} ({months[-1]} -> {months[0]}) newest-first")

ok, skip, fail = [], [], []
ok_since_sync = 0

for i, yyyymm in enumerate(months, 1):
    names = expected_files(PAIR, yyyymm)
    have = all((drive_processed / n).exists() or (repo_processed / n).exists() for n in names)

    if have:
        skip.append(yyyymm)
        if i % 10 == 0:
            print(f"[{i}/{len(months)}] skip {yyyymm} (already exists)")
        continue

    print(f"[{i}/{len(months)}] fetch {PAIR} {yyyymm}")
    try:
        _ = download_and_save(PAIR, yyyymm, compression="snappy")
        ok.append(yyyymm)
        ok_since_sync += 1

        # periodic sync to persistent storage
        if ok_since_sync >= SYNC_EVERY:
            sync_repo_to_drive()
            ok_since_sync = 0

    except Exception as e:
        print(f"  ERROR {yyyymm}: {e}")
        fail.append(yyyymm)

        # safety stop if endpoint is flaky
        if len(fail) >= MAX_FAIL_BEFORE_STOP:
            print(f"Stopping early: reached {MAX_FAIL_BEFORE_STOP} failures.")
            break

# final sync
sync_repo_to_drive()

print("\nDone")
print(f"Downloaded: {len(ok)}")
print(f"Skipped existing: {len(skip)}")
print(f"Failed: {len(fail)}")
if ok:
    print("Newest downloaded months:", ok[:10])   # because newest-first
if fail:
    print("Failed months:", fail[:20])

Synced Drive -> repo: 0 new file(s)
Planned months: 12 (202201 -> 202212) newest-first
[1/12] fetch EUR/USD 202212


INFO:DUKASCRIPT:current timestamp :2022-12-01T03:18:25.013000
INFO:DUKASCRIPT:current timestamp :2022-12-01T07:33:11.337000
INFO:DUKASCRIPT:current timestamp :2022-12-01T09:34:47.515000
INFO:DUKASCRIPT:current timestamp :2022-12-01T12:54:16.833000
INFO:DUKASCRIPT:current timestamp :2022-12-01T14:44:02.069000
INFO:DUKASCRIPT:current timestamp :2022-12-01T16:04:10.755000
INFO:DUKASCRIPT:current timestamp :2022-12-01T18:18:29.506000
INFO:DUKASCRIPT:current timestamp :2022-12-01T23:12:42.936000
INFO:DUKASCRIPT:current timestamp :2022-12-02T04:29:05.845000
INFO:DUKASCRIPT:current timestamp :2022-12-02T08:17:58.115000
INFO:DUKASCRIPT:current timestamp :2022-12-02T11:11:42.519000
INFO:DUKASCRIPT:current timestamp :2022-12-02T13:48:37.142000
INFO:DUKASCRIPT:current timestamp :2022-12-02T15:03:08.553000
INFO:DUKASCRIPT:current timestamp :2022-12-02T16:56:17.605000
INFO:DUKASCRIPT:current timestamp :2022-12-02T20:52:33.222000
INFO:DUKASCRIPT:current timestamp :2022-12-05T02:34:56.576000
INFO:DUK

  Received 3,531,569 ticks
Synced repo -> Drive: 2 new file(s)
[2/12] fetch EUR/USD 202211


INFO:DUKASCRIPT:current timestamp :2022-11-01T05:38:45.736000
INFO:DUKASCRIPT:current timestamp :2022-11-01T08:45:54.411000
INFO:DUKASCRIPT:current timestamp :2022-11-01T12:27:44.328000
INFO:DUKASCRIPT:current timestamp :2022-11-01T14:32:12.669000
INFO:DUKASCRIPT:current timestamp :2022-11-01T16:29:39.539000
INFO:DUKASCRIPT:current timestamp :2022-11-01T21:57:14.327000
INFO:DUKASCRIPT:current timestamp :2022-11-02T04:17:09.905000
INFO:DUKASCRIPT:current timestamp :2022-11-02T08:51:28.458000
INFO:DUKASCRIPT:current timestamp :2022-11-02T12:44:28.570000
INFO:DUKASCRIPT:current timestamp :2022-11-02T15:29:23.938000
INFO:DUKASCRIPT:current timestamp :2022-11-02T18:28:00.965000
INFO:DUKASCRIPT:current timestamp :2022-11-02T19:24:42.712000
INFO:DUKASCRIPT:current timestamp :2022-11-03T00:42:00.902000
INFO:DUKASCRIPT:current timestamp :2022-11-03T06:46:11.826000
INFO:DUKASCRIPT:current timestamp :2022-11-03T09:20:12.615000
INFO:DUKASCRIPT:current timestamp :2022-11-03T12:12:26.528000
INFO:DUK

  Received 4,428,942 ticks
Synced repo -> Drive: 2 new file(s)
[3/12] fetch EUR/USD 202210


...

  Received 2,702,726 ticks
Synced repo -> Drive: 2 new file(s)
[11/12] fetch EUR/USD 202202


INFO:DUKASCRIPT:current timestamp :2022-02-01T12:04:22.471000
INFO:DUKASCRIPT:current timestamp :2022-02-01T19:27:07.942000
INFO:DUKASCRIPT:current timestamp :2022-02-02T11:50:32.353000
INFO:DUKASCRIPT:current timestamp :2022-02-02T22:20:13.967000
INFO:DUKASCRIPT:current timestamp :2022-02-03T12:48:22.456000
INFO:DUKASCRIPT:current timestamp :2022-02-03T14:35:51.970000
INFO:DUKASCRIPT:current timestamp :2022-02-03T17:24:31.845000
INFO:DUKASCRIPT:current timestamp :2022-02-04T07:42:32.001000
INFO:DUKASCRIPT:current timestamp :2022-02-04T13:34:26.135000
INFO:DUKASCRIPT:current timestamp :2022-02-04T17:58:10.336000
INFO:DUKASCRIPT:current timestamp :2022-02-07T09:02:32.436000
INFO:DUKASCRIPT:current timestamp :2022-02-07T15:46:40.390000
INFO:DUKASCRIPT:current timestamp :2022-02-08T07:06:43.642000
INFO:DUKASCRIPT:current timestamp :2022-02-08T14:31:00.977000
INFO:DUKASCRIPT:current timestamp :2022-02-09T08:26:52.790000
INFO:DUKASCRIPT:current timestamp :2022-02-09T17:32:05.636000
INFO:DUK

  Received 1,737,911 ticks
Synced repo -> Drive: 2 new file(s)
[12/12] fetch EUR/USD 202201


INFO:DUKASCRIPT:current timestamp :2022-01-03T13:21:45.359000
INFO:DUKASCRIPT:current timestamp :2022-01-03T19:42:43.485000
INFO:DUKASCRIPT:current timestamp :2022-01-04T09:46:05.404000
INFO:DUKASCRIPT:current timestamp :2022-01-04T16:36:41.398000
INFO:DUKASCRIPT:current timestamp :2022-01-05T10:12:49.950000
INFO:DUKASCRIPT:current timestamp :2022-01-05T16:33:59.739000
INFO:DUKASCRIPT:current timestamp :2022-01-06T07:15:38.196000
INFO:DUKASCRIPT:current timestamp :2022-01-06T14:28:20.073000
INFO:DUKASCRIPT:current timestamp :2022-01-06T22:59:28.081000
INFO:DUKASCRIPT:current timestamp :2022-01-07T13:43:44.489000
INFO:DUKASCRIPT:current timestamp :2022-01-09T22:44:17.768000
INFO:DUKASCRIPT:current timestamp :2022-01-10T11:27:54.586000
INFO:DUKASCRIPT:current timestamp :2022-01-10T18:21:44.556000
INFO:DUKASCRIPT:current timestamp :2022-01-11T11:17:45.318000
INFO:DUKASCRIPT:current timestamp :2022-01-11T19:59:52.266000
INFO:DUKASCRIPT:current timestamp :2022-01-12T13:30:11.233000
INFO:DUK

  Received 1,398,880 ticks
Synced repo -> Drive: 2 new file(s)
Synced repo -> Drive: 0 new file(s)

Done
Downloaded: 12
Skipped existing: 0
Failed: 0
Newest downloaded months: ['202212', '202211', '202210', '202209', '202208', '202207', '202206', '202205', '202204', '202203']


In [23]:
# Manifest / health check
store = drive_processed if IN_COLAB else repo_processed
files = sorted(store.glob("*.parquet"))
print(f"Store: {store}")
print(f"Total parquet files: {len(files)}")

total_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
print(f"Total size: {total_mb:.1f} MB")

rows = []
for fp in files:
    name = fp.name
    # expected format: eurusd_dukascopy_bid_202101.parquet
    parts = name.replace(".parquet", "").split("_")
    if len(parts) >= 4:
        yyyymm = parts[-1]
        side = parts[-2]
    else:
        yyyymm = ""
        side = ""
    rows.append({"file": name, "yyyymm": yyyymm, "side": side, "mb": fp.stat().st_size / 1e6})

m = pd.DataFrame(rows)
if len(m):
    print("\nFiles by month (head):")
    display(m.sort_values(["yyyymm", "side"]).head(20))

    month_counts = m.groupby("yyyymm").size().rename("n_files").reset_index().sort_values("yyyymm")
    print("\nMonth coverage (last 24):")
    display(month_counts.tail(24))

# Note for gitignore/git pull workflow:
# - Keep parquet in Drive path above (not in git).
# - Notebook syncs between repo output and Drive cache.
# - After git pull/reclone, data remains in Drive and is re-synced.

Store: /content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed
Total parquet files: 108
Total size: 2096.9 MB

Files by month (head):


,file,yyyymm,side,mb
0,eurusd_dukascopy_ask_202101.parquet,202101,ask,18.387604
54,eurusd_dukascopy_bid_202101.parquet,202101,bid,18.384328
1,eurusd_dukascopy_ask_202102.parquet,202102,ask,15.310751
55,eurusd_dukascopy_bid_202102.parquet,202102,bid,15.227004
2,eurusd_dukascopy_ask_202201.parquet,202201,ask,13.255482
56,eurusd_dukascopy_bid_202201.parquet,202201,bid,13.107360
3,eurusd_dukascopy_ask_202202.parquet,202202,ask,16.104319
57,eurusd_dukascopy_bid_202202.parquet,202202,bid,16.167753
4,eurusd_dukascopy_ask_202203.parquet,202203,ask,25.474490
58,eurusd_dukascopy_bid_202203.parquet,202203,bid,25.454520



Month coverage (last 24):


,yyyymm,n_files
30,202405,2
31,202406,2
32,202407,2
33,202408,2
34,202409,2
35,202410,2
36,202411,2
37,202412,2
38,202501,2
39,202502,2


In [24]:
# End-of-HW0 summary: Drive processed folder
from pathlib import Path

store = drive_processed if IN_COLAB else repo_processed
files = sorted(store.glob("*.parquet"))

total_bytes = sum(f.stat().st_size for f in files)
n_files = len(files)

# Fast row counts (no full load)
try:
    import pyarrow.parquet as pq

    def nrows(fp):
        return pq.ParquetFile(fp).metadata.num_rows

    rows_all = sum(nrows(f) for f in files)
    bid_files = [f for f in files if "_bid_" in f.name.lower()]
    ask_files = [f for f in files if "_ask_" in f.name.lower()]
    rows_bid = sum(nrows(f) for f in bid_files)
    rows_ask = sum(nrows(f) for f in ask_files)
except Exception as e:
    print("pyarrow fast path failed, falling back to pandas (slower):", e)
    import pandas as pd

    rows_all = sum(len(pd.read_parquet(f, columns=["datetime"])) for f in files)
    bid_files = [f for f in files if "_bid_" in f.name.lower()]
    ask_files = [f for f in files if "_ask_" in f.name.lower()]
    rows_bid = sum(len(pd.read_parquet(f, columns=["datetime"])) for f in bid_files)
    rows_ask = sum(len(pd.read_parquet(f, columns=["datetime"])) for f in ask_files)

print("=" * 60)
print("HW0 — processed store summary")
print("=" * 60)
print(f"Folder: {store}")
print(f"Parquet files: {n_files}")
print(f"Total size: {total_bytes / (1024**3):.3f} GB  ({total_bytes / (1024**2):.1f} MB)")
print()
print(f"Rows (all files summed): {rows_all:,}")
print(f"Rows bid side:           {rows_bid:,}")
print(f"Rows ask side:           {rows_ask:,}")
print()
print("Approx. unique tick times (bid≈ask): use bid row count above.")
print("If bid+ask always paired, rows_all ≈ 2 × unique ticks.")
print("=" * 60)

HW0 — processed store summary
Folder: /content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed
Parquet files: 108
Total size: 2.048 GB  (2096.9 MB)

Rows (all files summed): 235,698,616
Rows bid side:           117,849,308
Rows ask side:           117,849,308

Approx. unique tick times (bid≈ask): use bid row count above.
If bid+ask always paired, rows_all ≈ 2 × unique ticks.
